In [ ]:
options(java.parameters = "-Xmx8g")
library(loadeR)
library(transformeR)
library(visualizeR) # spatialPlot
library(geoprocessoR)
library(climate4R.indices) # linearTrend

In [ ]:
library(magrittr)
library(sp)
library(RColorBrewer)
library(rgdal) # readOGR

In [ ]:
regs <- get(load("../reference-regions/IPCC-WGI-reference-regions-v4_R.rda")) %>% as("SpatialPolygons")

In [ ]:
regs.area <- c("NWN", "NEN", "WNA", "CNA","ENA","NCA") # North America regions

In [ ]:
regs <- regs[regs.area]
plot(regs)

In [ ]:
coast <- readOGR("auxiliary-material/WORLD_coastline.shp")

In [ ]:
UDG.datasets()$OBSERVATIONS

In [ ]:
lonLim <- c(168.0, -50)
latLim <- c(2.2, 85)

In [ ]:
years <- 1980:2014

In [ ]:
var <- "pr"

In [ ]:
grid <- loadGridData("auxiliary-material/W5E5_NorthAmerica_pr_1980-2014_yearly.nc4", var = "pr")

In [ ]:
grid <- setGridProj(grid = grid, proj = proj4string(regs))

In [ ]:
spatialPlot(climatology(grid),
            at = seq(0, 10, 1), 
            set.min = 0,
            set.max = 10,
            backdrop.theme = "coastline",
            col.regions = brewer.pal(n = 9, "BuPu") %>% colorRampPalette(),
            main = paste0("Climatology of total precipitation (W5E5, ", min(years), "-", max(years), ") - mm/day"),
            sp.layout = list(
                list(regs, first = FALSE, lwd = 0.6),
                list("sp.text", coordinates(regs), names(regs), first = FALSE, cex = 1)
))

In [ ]:
grid %<>% overGrid(regs)

In [ ]:
spatialPlot(climatology(grid),
            at = seq(0, 10, 1), 
            set.min = 0,
            set.max = 10,
            backdrop.theme = "coastline",
            col.regions = brewer.pal(n = 9, "BuPu") %>% colorRampPalette(),
            main = paste0("Climatology of total precipitation (W5E5, ", min(years), "-", max(years), ") - mm/day"),
            sp.layout = list(
                list(regs, first = FALSE, lwd = 0.6),
                list("sp.text", coordinates(regs), names(regs), first = FALSE, cex = 1)
))

In [ ]:
trendGrid <- linearTrend(grid) %>% subsetGrid(var = "b")

In [ ]:
spatialPlot(trendGrid, 
            col.regions = brewer.pal(n = 9, "BrBG") %>% colorRampPalette(),
            at = seq(-0.1, 0.1, 0.01), 
            set.min = -0.1,
            set.max = 0.1,
            main = paste("Linear trends of the annual total precipitaion (mm/day) - W5E5"),
            sp.layout = list(  
            list(regs, first = FALSE, lwd = 0.6),
            list(coast, col = "gray50", first = FALSE, lwd = 0.6),  
            list("sp.text", coordinates(regs), names(regs), first = FALSE, cex = 1)
))

In [ ]:
pvalGrid <- linearTrend(grid) %>% subsetGrid(var = "pval")
pvalGrid <- binaryGrid(pvalGrid, threshold = 0.1, condition = "GT")

In [ ]:
spatialPlot(pvalGrid)

In [ ]:
mask <- binaryGrid(climatology(grid),condition = "GE", threshold = 0, values = c(NA,1))
pvalGrid <- gridArithmetics(pvalGrid, mask)

In [ ]:
spatialPlot(pvalGrid)

In [ ]:
hatching.lines <- lapply(c("45","-45"), FUN = function(angle) {
  c(map.hatching(clim = climatology(pvalGrid), 
                 threshold = 0.5, 
                 condition = "GE", 
                 density = 4,
                 angle = angle, coverage.percent = 50,
                 upscaling.aggr.fun = list(FUN = "mean", na.rm = TRUE)
  ), 
  "which" = 1, lwd = 0.5)
})

In [ ]:
spatialPlot(trendGrid, 
            col.regions = brewer.pal(n = 9, "BrBG") %>% colorRampPalette(),
            at = seq(-0.1, 0.1, 0.01), 
            set.min = -0.1,
            set.max = 0.1,
            main = paste("Linear trend of total precipitation (mm/day) - W5E5"),
            sp.layout = list(
              hatching.lines[[1]],
              hatching.lines[[2]],  
              list(regs, first = FALSE, lwd = 0.6),
              list(coast, col = "gray50", first = FALSE, lwd = 0.6),  
              list("sp.text", coordinates(regs), names(regs), first = FALSE, cex = 1)
            )
)

In [ ]:
grid.regs <- lapply(names(regs), function(r) overGrid(grid, regs[r]))

In [ ]:
spatialPlot(climatology(grid.regs[["NWN"]]), 
            col.regions = brewer.pal(n = 9, "BuPu") %>% colorRampPalette()
)

In [ ]:
spatial.mean <- function(grid) aggregateGrid(grid, aggr.spatial = list(FUN = "mean", na.rm = TRUE)) %>% scaleGrid(type = "center")
grid.anom <- lapply(grid.regs, spatial.mean)

In [ ]:
names(grid.anom)

In [ ]:
trend.val <- lapply(1:length(grid.anom), FUN = function(x) {
    aux <- linearTrend(grid.anom[[x]]) %>% subsetGrid(var = "b")
    round(aux$Data[1], digits = 3)
})
names(grid.anom) <- sprintf("%s (%g mm/day/yr)", names(grid.anom), trend.val)

In [ ]:
names(grid.anom)

In [ ]:
temporalPlot(grid.anom,
               xyplot.custom = list(
                 main = "Precipitation anomaly time series (mm/day)", ylim=c(-0.65, 0.65)
               ))

In [ ]:
sessionInfo()